# Autoresearch-astro Experiment Analysis

Analysis of autonomous streaming-throughput results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 6 columns:
# commit, rows_per_sec, mb_per_sec, peak_rss_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["rows_per_sec"] = pd.to_numeric(df["rows_per_sec"], errors="coerce")
df["mb_per_sec"] = pd.to_numeric(df["mb_per_sec"], errors="coerce")
df["peak_rss_gb"] = pd.to_numeric(df["peak_rss_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(
        f"  #{i:3d}  {row['rows_per_sec']:8.2f} rows/s  "
        f"{row['mb_per_sec']:6.1f} MB/s  mem={row['peak_rss_gb']:.1f}GB  {row['description']}"
    )

## Throughput Over Time

Track how the best (kept) rows_per_sec evolves as experiments progress. The running maximum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_rps = valid.loc[0, "rows_per_sec"]

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["rows_per_sec"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["rows_per_sec"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_rps = valid.loc[kept_mask, "rows_per_sec"]
running_max = kept_rps.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Baseline reference
ax.axhline(baseline_rps, color="#888888", linestyle="--", linewidth=1,
           alpha=0.7, zorder=1, label=f"Baseline ({baseline_rps:.2f} rows/s)")

# Label each kept experiment with its description
for idx, rps in zip(kept_idx, kept_rps):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, rps),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Streaming throughput, rows/sec (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to just above best
best_rps = kept_rps.max()
margin = (best_rps - baseline_rps) * 0.15 or 1.0
ax.set_ylim(baseline_rps - margin, best_rps + margin * 3)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_rps = df.iloc[0]["rows_per_sec"]
best_rps = kept["rows_per_sec"].max()
best_row = kept.loc[kept["rows_per_sec"].idxmax()]

print(f"Baseline rows_per_sec: {baseline_rps:.2f}")
print(f"Best rows_per_sec:     {best_rps:.2f}")
print(f"Speedup:               {best_rps / baseline_rps:.2f}x")
print(f"Best experiment:       {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: {row['rows_per_sec']:8.2f} rows/s  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's throughput
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_rps"] = kept["rows_per_sec"].shift(1)
kept["delta"] = kept["rows_per_sec"] - kept["prev_rps"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>9}  {'rows/s':>9}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+9.2f}  {row['rows_per_sec']:9.2f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+9.2f}  {'':>9}  TOTAL improvement over baseline")